# PIPELINE-NAME — run it

Fill in the three lines in the next cell, then **Run → Run All Cells**. You'll get an email when the job finishes. Nothing else on this page needs editing.

In [ ]:
# ---- 1. FILL THESE IN -------------------------------------------------
INPUT_DIR  = "example/input"                    # folder with your input files
OUTPUT_DIR = "/blue/GROUP/YOUR_USERNAME/PIPELINE-NAME-results"   # where results go
EMAIL      = "you@ufl.edu"                      # for the "job finished" email
# ----------------------------------------------------------------------
# Leave INPUT_DIR as-is the first time to run the bundled example.

## 2. Check inputs (automatic)

In [ ]:
import os, subprocess, sys, datetime
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
os.chdir(repo)

inp = Path(INPUT_DIR).expanduser()
if not inp.is_absolute():
    inp = repo / inp
assert inp.is_dir(), f"INPUT_DIR not found: {inp}"
files = [p for p in inp.iterdir() if p.is_file()]
assert files, f"INPUT_DIR is empty: {inp}"
assert "@" in EMAIL, "EMAIL doesn't look like an email address"
out = Path(OUTPUT_DIR).expanduser()
out.mkdir(parents=True, exist_ok=True)
print(f"OK: {len(files)} input file(s); results will go to {out}")

## 3. Submit the job (automatic)

In [ ]:
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
cfg_path = repo / "logs" / f"config-{stamp}.yml"
cfg_path.parent.mkdir(exist_ok=True)
cfg_path.write_text(f"input_dir: {inp}\noutput_dir: {out}\n")

cmd = ["sbatch", f"--mail-user={EMAIL}", f"--export=ALL,CONFIG={cfg_path}", "slurm/run.slurm"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("sbatch failed:\n", res.stderr)
    print("If you're not on HiPerGator, this is expected — the pipeline only runs on the cluster.")
else:
    print(res.stdout.strip())
    print(f"Logs: {repo/'logs'}   Results (when done): {out}")

## 4. Check progress (optional)

Run this cell any time. `PD` = waiting in queue, `R` = running, nothing listed = finished.

In [ ]:
print(subprocess.run(["squeue", "--me", "--name=PIPELINE-NAME"], capture_output=True, text=True).stdout or "(squeue not available here)")